In [ ]:
# !sudo apt-get install -y fonts-nanum
# 폰트 설치 후 Colab 메뉴에서 런타임/세션 다시 시작을 선택합니다.

In [ ]:
# !pip install -U openai

In [1]:
# from google.colab import userdata
import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수에서 가져오기
api_key = os.getenv("OPENAI_API_KEY")

# Colab에 저장한 Secret에서 API 키 가져와 환경 변수에 등록
os.environ["OPENAI_API_KEY"] = api_key

# Colab에 저장한 Secret에서 API 키 가져와 환경 변수에 등록
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

import openai

client = openai.OpenAI()

In [2]:
def make_evaluation_prompt(prompt, answer):
    return f"""
        프롬프트: {prompt}
        응답: {answer}

        위 프롬프트에 대한 응답을 정답/오답으로 답변해주세요.
        정답/오답만 답변합니다.
    """

In [ ]:
prompts_and_answers = [
    ("프랑스의 수도는 어디인가요?", "파리"),
    ("전세계에서 가장 높은 산은 어디인가요?", "에베레스트"),
    ("1 더하기 3은 얼마인가요?", "5")
]

for prompt, answer in prompts_and_answers:
    eval_prompt = make_evaluation_prompt(prompt, answer)

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": eval_prompt}
        ]
    )

    content = response.choices[0].message.content
    print(f"Prompt: {prompt}, Answer: {answer}, Evaluation: {content}")

    if content == '오답':
        score = {"correct": False}
    else:
        score = {"correct": True}
    # You can do something with the score here, e.g., store it in a list
    print(f"Score: {score}")

In [ ]:
results = []
for prompt, answer in prompts_and_answers:
    # 프롬프트에 대한 답변이 맞았는지 평가를 요청
    eval_prompt = make_evaluation_prompt(prompt, answer)  # 프롬프트와 응답으로 평가 프롬프트 생성
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": eval_prompt}
        ]
    )

    content = response.choices[0].message.content
    print(content)

    if content == '오답':
        score = {"correct": False}
    else:
        score = {"correct": True}
    score["prompt"] = prompt  # 원본 프롬프트나 식별자 저장
    score["answer"] = answer
    score["model"] = "gpt-4o-mini"
    results.append(score)

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df  # 데이터프레임 출력